C14. Потолочные эффекты рубрики и согласованность рангов (Spagliardi 2026, ATLAS 2026).....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/evaluation_robustness')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
ITEMS, POS = load_items()
print()
ARMS = sorted(ITEMS)
V = pd.DataFrame({a: [int(ITEMS[a][k]['met']) for k in POS] for a in ARMS}, index=pd.MultiIndex.from_tuples(POS, names=['case', 'cid']))
PTS = pd.Series([ITEMS[ARMS[0]][k]['points'] for k in POS], index=V.index, name='points')
AX = pd.Series([ITEMS[ARMS[0]][k]['axis'] for k in POS], index=V.index, name='axis')
print('матрица вердиктов: %d пунктов × %d плеч' % V.shape)


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


плеч с попунктными вердиктами: 18
общих положительных пунктов: 2249

матрица вердиктов: 2249 пунктов × 18 плеч


In [2]:
rate = V.mean(axis=1)
tot = (V * PTS.to_numpy()[:, None]).groupby(level=0).sum().sum(axis=0)
disc = V.apply(lambda col: np.nan, axis=0)
armscore = V.mean(axis=0)
disc = V.apply(lambda row: np.corrcoef(row.to_numpy(float), armscore.to_numpy())[0, 1], axis=1)
INFO = pd.DataFrame({'доля плеч': rate, 'дискриминативность': disc, 'points': PTS, 'axis': AX})
bins = pd.cut(INFO['доля плеч'], [-0.001, 0.05, 0.25, 0.75, 0.95, 1.001], labels=['пол (<5%)', 'трудные', 'средние', 'лёгкие', 'потолок (>95%)'])
INFO['зона'] = bins
show(INFO.groupby('зона', observed=True).agg(пунктов=('доля плеч', 'size'), ср_доля=('доля плеч', 'mean'), ср_дискрим=('дискриминативность', 'mean'), ср_вес=('points', 'mean')), 'Зоны информативности пунктов рубрики')
n_ceil = int((INFO['зона'] == 'потолок (>95%)').sum())
n_floor = int((INFO['зона'] == 'пол (<5%)').sum())
print()
print('потолочных пунктов: %d (%.1f%%), половых: %d (%.1f%%)' % (n_ceil, 100 * n_ceil / len(INFO), n_floor, 100 * n_floor / len(INFO)))
print('они дают %.1f%% всего веса рубрики' % (100 * INFO.loc[INFO['зона'].isin(['потолок (>95%)', 'пол (<5%)']), 'points'].sum() / INFO['points'].sum()))



Зоны информативности пунктов рубрики
------------------------------------------------------------
                пунктов  ср_доля  ср_дискрим  ср_вес
зона                                                
пол (<5%)           109   0.0000         NaN  5.0092
трудные             363   0.1370      0.2801  5.5482
средние             710   0.4857      0.4283  6.0408
лёгкие              456   0.8799      0.3208  6.8092
потолок (>95%)      611   1.0000         NaN  7.7938

потолочных пунктов: 611 (27.2%), половых: 109 (4.8%)
они дают 36.1% всего веса рубрики


In [3]:
def arm_scores(mask):
    sub, p = (V[mask], PTS[mask])
    return (sub.T * p.to_numpy()).T.sum() / p.sum()
full = arm_scores(pd.Series(True, index=V.index))
no_ceil = arm_scores(INFO['зона'] != 'потолок (>95%)')
no_both = arm_scores(~INFO['зона'].isin(['потолок (>95%)', 'пол (<5%)']))
mid = arm_scores(INFO['доля плеч'].between(0.2, 0.8))
R = pd.DataFrame({'все пункты': full, 'без потолка': no_ceil, 'без потолка и пола': no_both, 'только середина 20-80%': mid})
R['ранг все'] = R['все пункты'].rank(ascending=False)
R['ранг середина'] = R['только середина 20-80%'].rank(ascending=False)
R['сдвиг ранга'] = R['ранг все'] - R['ранг середина']
show(R.sort_values('все пункты', ascending=False), 'Балл плеча при разных наборах пунктов')
for c in ['без потолка', 'без потолка и пола', 'только середина 20-80%']:
    rho, p = stats.spearmanr(R['все пункты'], R[c])
    print('ранговая корреляция «все пункты» и «%s»: rho = %.4f (p = %s)' % (c, rho, fmt_p(p)))



Балл плеча при разных наборах пунктов
------------------------------------------------------------
                        все пункты  без потолка  без потолка и пола  только середина 20-80%  ранг все  ранг середина  сдвиг ранга
P-strong                    0.8080       0.7161              0.7577                  0.7858    1.0000         1.0000       0.0000
P-base                      0.7891       0.6883              0.7282                  0.7456    2.0000         2.0000       0.0000
P-mixed                     0.7891       0.6882              0.7281                  0.7376    3.0000         3.0000       0.0000
homog deepseekv4            0.7814       0.6768              0.7161                  0.7325    4.0000         4.0000       0.0000
homog gpt                   0.7620       0.6482              0.6858                  0.6570    5.0000         5.0000       0.0000
P-weak                      0.7505       0.6312              0.6678                  0.6555    6.0000         6.0000    

In [4]:
MEM = COMPOSITION['P-base']
OUTS = [m for m in ['deepseekv4pro', 'gemini', 'glm', 'kimi']]
ext_cols = ['baseline ' + m for m in OUTS if 'baseline ' + m in V.columns]
diff_ext = 1 - V[ext_cols].mean(axis=1)
mem_cols = ['baseline ' + m for m in MEM if 'baseline ' + m in V.columns]
best_member = V[mem_cols].max(axis=1)
panel = V['P-base']
strata = pd.cut(diff_ext, [-0.001, 0.001, 0.25, 0.5, 0.75, 1.001], labels=['очень лёгкие', 'лёгкие', 'средние', 'трудные', 'очень трудные'])

def mh_table(mask, label):
    rows = []
    for s in strata.cat.categories:
        sel = (strata == s) & mask
        if sel.sum() < 10:
            continue
        p_, b_ = (panel[sel], best_member[sel])
        n10 = int(((p_ == 1) & (b_ == 0)).sum())
        n01 = int(((p_ == 0) & (b_ == 1)).sum())
        orr = n10 / n01 if n01 else np.inf
        pv = float(stats.binomtest(n10, n10 + n01, 0.5).pvalue) if n10 + n01 else np.nan
        rows.append({'набор': label, 'страта': s, 'пунктов': int(sel.sum()), 'панель+ лучший−': n10, 'панель− лучший+': n01, 'OR': orr, 'p (McNemar)': pv})
    return pd.DataFrame(rows)
T = pd.concat([mh_table(pd.Series(True, index=V.index), 'все пункты'), mh_table(INFO['зона'] != 'потолок (>95%)', 'без потолка')], ignore_index=True)
show(T.set_index(['набор', 'страта']), 'H3.7 с потолочными пунктами и без них')
print()
print('Если в строке «очень лёгкие» после удаления потолка расхождений почти')
print('не остаётся, вывод «панель теряет на лёгких пунктах» держится на')
print('считанных наблюдениях внутри зоны, где рубрика не различает системы.')



H3.7 с потолочными пунктами и без них
------------------------------------------------------------
                           пунктов  панель+ лучший−  панель− лучший+     OR  p (McNemar)
набор       страта                                                                      
все пункты  очень лёгкие       727                1                9 0.1111       0.0215
            лёгкие             267                4               12 0.3333       0.0768
            средние            234               17               35 0.4857       0.0175
            трудные            339               53               49 1.0816       0.7666
            очень трудные      682              171               74 2.3108       0.0000
без потолка очень лёгкие       116                1                9 0.1111       0.0215
            лёгкие             267                4               12 0.3333       0.0768
            средние            234               17               35 0.4857       0.0175
          

In [5]:
long = V.stack().rename('met').reset_index()
long.columns = ['case', 'cid', 'arm', 'met']
long['item'] = long['case'].astype(str) + '#' + long['cid'].astype(str)
keep = long.groupby('item')['met'].transform(lambda s: 0 < s.sum() < len(s))
sub = long[keep].copy()
print('пунктов, различающих плечи (не все 0 и не все 1): %d из %d' % (sub['item'].nunique(), long['item'].nunique()))
Xa = pd.get_dummies(sub['arm'], prefix='a', drop_first=True).astype(float)
Xi = pd.get_dummies(sub['item'], prefix='i', drop_first=True).astype(float)
mod = sm.Logit(sub['met'].astype(float), sm.add_constant(pd.concat([Xa, Xi], axis=1)))
fit = mod.fit(disp=0, method='lbfgs', maxiter=200)
theta = {c[2:]: fit.params[c] for c in Xa.columns}
ref = [a for a in ARMS if a not in theta][0]
theta[ref] = 0.0
TH = pd.Series(theta).sort_values(ascending=False).rename('θ (Раш)')
cmp_ = pd.DataFrame({'θ (Раш)': TH, 'сырая доля': full})
cmp_['ранг θ'] = cmp_['θ (Раш)'].rank(ascending=False)
cmp_['ранг сырой'] = cmp_['сырая доля'].rank(ascending=False)
cmp_['|сдвиг|'] = (cmp_['ранг θ'] - cmp_['ранг сырой']).abs()
show(cmp_.sort_values('θ (Раш)', ascending=False), 'Латентная способность против сырой доли (база сравнения: %s)' % ref)
rho, p = stats.spearmanr(cmp_['θ (Раш)'], cmp_['сырая доля'])
print('\nранговая корреляция θ и сырой доли: rho = %.4f (p = %s)' % (rho, fmt_p(p)))
print('Высокая rho означает, что сырой рейтинг плеч НЕ артефакт композиции')
print('пунктов — это и есть проверка, ради которой Раш здесь применяется.')


пунктов, различающих плечи (не все 0 и не все 1): 1529 из 2249



Латентная способность против сырой доли (база сравнения: P-base)
----------------------------------------------------------------
                        θ (Раш)  сырая доля  ранг θ  ранг сырой  |сдвиг|
P-strong                 0.1982      0.8080  1.0000      1.0000   0.0000
P-base                   0.0000      0.7891  2.0000      2.0000   0.0000
P-mixed                 -0.0194      0.7891  3.0000      3.0000   0.0000
homog deepseekv4        -0.0540      0.7814  4.0000      4.0000   0.0000
homog gpt               -0.2668      0.7620  5.0000      5.0000   0.0000
P-weak                  -0.4694      0.7505  6.0000      6.0000   0.0000
homog gemini            -0.8492      0.7192  7.0000      7.0000   0.0000
homog qwen              -1.0110      0.7011  8.0000      8.0000   0.0000
baseline deepseekv4     -1.6080      0.6489  9.0000      9.0000   0.0000
USC@5 gpt               -1.6668      0.6402 10.0000     10.0000   0.0000
USC@7 gpt               -1.7166      0.6370 11.0000     12.0000   

In [6]:
rng = np.random.default_rng(0)
rhos, drops = ([], [])
for _ in range(200):
    m = rng.random(len(V)) < 0.5
    a1 = arm_scores(pd.Series(m, index=V.index))
    a2 = arm_scores(pd.Series(~m, index=V.index))
    rhos.append(stats.spearmanr(a1, a2).statistic)
    drops.append(float((a1.rank(ascending=False) - a2.rank(ascending=False)).abs().mean()))
print('split-half ранговая корреляция: медиана %.4f  (5%%-квантиль %.4f)' % (np.median(rhos), np.quantile(rhos, 0.05)))
print('средний сдвиг ранга между половинами: медиана %.2f позиции' % np.median(drops))
print()
print('Формула Спирмена-Брауна для полной длины: %.4f' % (2 * np.median(rhos) / (1 + np.median(rhos))))


split-half ранговая корреляция: медиана 0.9773  (5%-квантиль 0.9587)
средний сдвиг ранга между половинами: медиана 0.67 позиции

Формула Спирмена-Брауна для полной длины: 0.9885


In [7]:
save(INFO.reset_index(), 'rubric_ceiling_and_rank_stability_item_info.csv')
save(R.reset_index(), 'rubric_ceiling_and_rank_stability_rank_stability.csv')
save(T, 'rubric_ceiling_and_rank_stability_h37_no_ceiling.csv')
save(cmp_.reset_index(), 'rubric_ceiling_and_rank_stability_rasch.csv')


сохранено: out/rubric_ceiling_and_rank_stability_item_info.csv  (2249 строк)
сохранено: out/rubric_ceiling_and_rank_stability_rank_stability.csv  (18 строк)
сохранено: out/rubric_ceiling_and_rank_stability_h37_no_ceiling.csv  (10 строк)
сохранено: out/rubric_ceiling_and_rank_stability_rasch.csv  (18 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\evaluation_robustness\\results\\rubric_ceiling_and_rank_stability_rasch.csv'